In [2]:
import pandas as pd
import numpy as np
import os
import re

# ============================================================
# CONFIG
# ============================================================

INPUT_DIR = "/Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks"
OUTPUT_DIR = "/Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# We need enough history for lag_28 / rolling_28
HISTORY_DAYS = 35


# ============================================================
# GET FILES
# ============================================================

files = [
    f for f in os.listdir(INPUT_DIR)
    if f.endswith(".parquet")
]

def extract_number(filename):
    match = re.search(r"\d+", filename)
    return int(match.group()) if match else 0

files = sorted(files, key=extract_number)

print(f"Found {len(files)} files")


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def create_features(df):

    df = df.copy()

    # Make sure date is datetime
    df["date"] = pd.to_datetime(df["date"])

    # Sort
    df = df.sort_values(["id", "date"])

    # --------------------------------------------------------
    # CALENDAR FEATURES
    # --------------------------------------------------------

    df["dayofweek"] = df["date"].dt.dayofweek
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype("int16")
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year

    df["is_weekend"] = (
        df["dayofweek"] >= 5
    ).astype("int8")

    df["dow_sin"] = np.sin(
        2 * np.pi * df["dayofweek"] / 7
    )

    df["dow_cos"] = np.cos(
        2 * np.pi * df["dayofweek"] / 7
    )

    # --------------------------------------------------------
    # LAGS
    # --------------------------------------------------------

    g = df.groupby("id")["sales"]

    df["lag_1"] = g.shift(1)
    df["lag_7"] = g.shift(7)
    df["lag_14"] = g.shift(14)
    df["lag_21"] = g.shift(21)
    df["lag_28"] = g.shift(28)

    # --------------------------------------------------------
    # ROLLING FEATURES
    # Shift first so today's sales are NEVER included.
    # --------------------------------------------------------

    shifted = g.shift(1)

    df["rolling_mean_7"] = (
        shifted.groupby(df["id"])
        .rolling(7)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["rolling_mean_28"] = (
        shifted.groupby(df["id"])
        .rolling(28)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["rolling_std_7"] = (
        shifted.groupby(df["id"])
        .rolling(7)
        .std()
        .reset_index(level=0, drop=True)
    )

    df["rolling_std_28"] = (
        shifted.groupby(df["id"])
        .rolling(28)
        .std()
        .reset_index(level=0, drop=True)
    )

    # --------------------------------------------------------
    # ZERO DEMAND
    # --------------------------------------------------------

    df["zero_rate_7"] = (
        shifted.eq(0)
        .groupby(df["id"])
        .rolling(7)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["zero_rate_28"] = (
        shifted.eq(0)
        .groupby(df["id"])
        .rolling(28)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # --------------------------------------------------------
    # DAYS SINCE LAST SALE
    # --------------------------------------------------------

    sale_date = df["date"].where(df["sales"] > 0)

    last_sale = (
        sale_date.groupby(df["id"])
        .ffill()
    )

    df["days_since_last_sale"] = (
        df["date"] - last_sale
    ).dt.days

    # Current day's sale must not influence this feature
    df["days_since_last_sale"] = (
        df.groupby("id")["days_since_last_sale"]
        .shift(1)
    )

    # --------------------------------------------------------
    # MOMENTUM / TREND
    # --------------------------------------------------------

    df["trend_1_7"] = (
        df["lag_1"] - df["lag_7"]
    )

    df["trend_7_28"] = (
        df["lag_7"] - df["lag_28"]
    )

    df["momentum_1_7"] = (
        df["lag_1"] /
        (df["lag_7"] + 1)
    )

    df["momentum_7_28"] = (
        df["lag_7"] /
        (df["lag_28"] + 1)
    )

    # --------------------------------------------------------
    # COEFFICIENT OF VARIATION
    # --------------------------------------------------------

    df["rolling_cv_28"] = (
        df["rolling_std_28"] /
        (df["rolling_mean_28"] + 1)
    )

    # --------------------------------------------------------
    # PRICE FEATURES
    # --------------------------------------------------------

    if "sell_price" in df.columns:

        df["price_change"] = (
            df.groupby("id")["sell_price"]
            .pct_change()
        )

        price_lag_7 = (
            df.groupby("id")["sell_price"]
            .shift(7)
        )

        df["price_to_lag_7"] = (
            df["sell_price"] /
            (price_lag_7 + 1e-6)
        )

        df["price_mean_7"] = (
            df.groupby("id")["sell_price"]
            .transform(
                lambda x: x.shift(1).rolling(7).mean()
            )
        )

    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    if "event_name_1" in df.columns:

        df["event_name_1"] = (
            df["event_name_1"]
            .fillna("NoEvent")
            .astype(str)
        )

        df["is_event"] = (
            df["event_name_1"] != "NoEvent"
        ).astype("int8")

    if "event_type_1" in df.columns:

        df["event_type_1"] = (
            df["event_type_1"]
            .fillna("NoEvent")
            .astype(str)
        )

    # --------------------------------------------------------
    # PRICE × EVENT
    # --------------------------------------------------------

    if "sell_price" in df.columns and "is_event" in df.columns:

        df["price_x_event"] = (
            df["sell_price"] *
            df["is_event"]
        )

    # --------------------------------------------------------
    # LAG × SNAP
    # --------------------------------------------------------

    if "snap_CA" in df.columns:

        df["lag_7_x_snap"] = (
            df["lag_7"] *
            df["snap_CA"]
        )

    # --------------------------------------------------------
    # REMOVE FEATURES WE DON'T WANT
    # --------------------------------------------------------

    remove_columns = [
        "cat_sales_mean",
        "store_sales_mean",
        "rel_to_cat",
        "rel_to_store",
        "price_norm"
    ]

    df.drop(
        columns=[
            c for c in remove_columns
            if c in df.columns
        ],
        inplace=True
    )

    return df


# ============================================================
# PROCESS CHUNKS
# ============================================================

history = None

for i, file in enumerate(files):

    print(f"\nProcessing {i + 1}/{len(files)}: {file}")

    input_path = os.path.join(INPUT_DIR, file)

    current = pd.read_parquet(input_path)

    current["date"] = pd.to_datetime(current["date"])

    # --------------------------------------------------------
    # ADD PREVIOUS HISTORY
    # --------------------------------------------------------

    if history is not None:

        working = pd.concat(
            [history, current],
            ignore_index=True
        )

    else:

        working = current.copy()

    # --------------------------------------------------------
    # FEATURE ENGINEERING
    # --------------------------------------------------------

    working = create_features(working)

    # --------------------------------------------------------
    # KEEP ONLY CURRENT CHUNK
    # --------------------------------------------------------

    current_dates = set(current["date"])

    output = working[
        working["date"].isin(current_dates)
    ].copy()

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    output_path = os.path.join(
        OUTPUT_DIR,
        f"part_{i}.parquet"
    )

    output.to_parquet(
        output_path,
        index=False
    )

    print(
        f"Saved {len(output):,} rows → {output_path}"
    )

    # --------------------------------------------------------
    # KEEP HISTORY
    #
    # Keep last 35 calendar days.
    # --------------------------------------------------------

    max_date = working["date"].max()

    history = working[
        working["date"] >=
        max_date - pd.Timedelta(days=HISTORY_DAYS)
    ].copy()

    # Remove previously generated feature columns
    # so the next iteration starts from raw historical
    # sales rather than recursively carrying features.
    #
    # Keep the original data columns only.
    feature_columns = [
        "dayofweek",
        "week_of_year",
        "month",
        "year",
        "is_weekend",
        "dow_sin",
        "dow_cos",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_21",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28",
        "zero_rate_7",
        "zero_rate_28",
        "days_since_last_sale",
        "trend_1_7",
        "trend_7_28",
        "momentum_1_7",
        "momentum_7_28",
        "rolling_cv_28",
        "price_change",
        "price_to_lag_7",
        "price_mean_7",
        "is_event",
        "price_x_event",
        "lag_7_x_snap"
    ]

    history = history.drop(
        columns=[
            c for c in feature_columns
            if c in history.columns
        ],
        errors="ignore"
    )


print("\n===================================")
print("FEATURE ENGINEERING COMPLETE")
print("===================================")

Found 69 files

Processing 1/69: part_0.parquet


/var/folders/3g/yctzkss146qd1vt6nc2yg9p40000gn/T/ipykernel_54470/958302641.py:199: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_0.parquet

Processing 2/69: part_1.parquet


/var/folders/3g/yctzkss146qd1vt6nc2yg9p40000gn/T/ipykernel_54470/958302641.py:199: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_1.parquet

Processing 3/69: part_2.parquet


...

Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_39.parquet

Processing 41/69: part_40.parquet


/var/folders/3g/yctzkss146qd1vt6nc2yg9p40000gn/T/ipykernel_54470/958302641.py:199: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_40.parquet

Processing 42/69: part_41.parquet


/var/folders/3g/yctzkss146qd1vt6nc2yg9p40000gn/T/ipykernel_54470/958302641.py:199: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_41.parquet

Processing 43/69: part_42.parquet
Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_42.parquet

Processing 44/69: part_43.parquet
Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_43.parquet

Processing 45/69: part_44.parquet
Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_44.parquet

Processing 46/69: part_45.parquet
Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_45.parquet

Processing 47/69: part_46.parquet
Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_46.parquet

Processing 48/69: part_47.parquet
Saved 853,720 rows → /Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_47.parquet

Processing 49/69: part_48.parquet
Saved 853,720 rows → /Users/abhigyanseth/

In [4]:
temp = pd.read_parquet("/Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_0.parquet")

print(temp.shape)

print("\nColumns:")
print(temp.columns.tolist())

print("\nNaN ratios:")
print(
    temp.isna().mean()
    .sort_values(ascending=False)
    .head(20)
)

print("\nDuplicate id/date:")
print(
    temp.duplicated(["id", "date"]).sum()
)

(853720, 46)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'dayofweek', 'week_of_year', 'month', 'year', 'is_weekend', 'dow_sin', 'dow_cos', 'lag_1', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'zero_rate_7', 'zero_rate_28', 'days_since_last_sale', 'trend_1_7', 'trend_7_28', 'momentum_1_7', 'momentum_7_28', 'rolling_cv_28', 'price_change', 'price_to_lag_7', 'price_mean_7', 'is_event', 'price_x_event', 'lag_7_x_snap']

NaN ratios:
lag_28                  1.000000
rolling_mean_28         1.000000
rolling_cv_28           1.000000
momentum_7_28           1.000000
rolling_std_28          1.000000
trend_7_28              1.000000
zero_rate_28            0.964286
lag_21                  0.750000
days_since_last_sale    0.636640
lag_14                  0.500000
price_to_lag_7          0.250

In [6]:
temp = temp.sort_values(["id", "date"])

expected = (
    temp.groupby("id")["sales"]
    .shift(7)
)

mask = (
    temp["lag_7"].notna() &
    expected.notna()
)

print(
    "Lag-7 mismatches:",
    (
        temp.loc[mask, "lag_7"].values
        != expected.loc[mask].values
    ).sum()
)

Lag-7 mismatches: 0


In [10]:
temp = pd.read_parquet("/Users/abhigyanseth/Projects/m5_forecasting/data/processed_data3/part_5.parquet")

temp = temp.sort_values(["id", "date"])

expected = (
    temp.groupby("id")["sales"]
    .shift(7)
)

mask = (
    temp["lag_7"].notna() &
    expected.notna()
)

print(
    "Lag-7 mismatches:",
    (
        temp.loc[mask, "lag_7"].values
        != expected.loc[mask].values
    ).sum()
)

Lag-7 mismatches: 0
